In [2]:
import pandas as pd
import sys
sys.path.append('../Source')

from Pipeline.build_comparisons import buildComparisons

alignedData = pd.read_csv('../Data/Processed/alignedData.csv')
modelingData = buildComparisons(alignedData)

print(f'Shape: {modelingData.shape[0]} rows, {modelingData.shape[1]} columns')
modelingData.head()





Shape: 2346 rows, 5 columns


,week,regionI,regionJ,winsI,total
0,2018-01-01/2018-01-07,CaucasusCA,Europe,1050,1505
1,2018-01-01/2018-01-07,CaucasusCA,MiddleEast,1050,3660
2,2018-01-01/2018-01-07,CaucasusCA,NorthAfrica,1050,1320
3,2018-01-01/2018-01-07,Europe,MiddleEast,455,3065
4,2018-01-01/2018-01-07,Europe,NorthAfrica,455,725


In [3]:
import pandas as pd
import numpy as np
import arviz as az 

In [4]:
regions = ['CaucasusCA', 'Europe', 'MiddleEast', 'NorthAfrica']
regionMap = {'CaucasusCA' : 0, 'Europe' : 1, 'MiddleEast' : 2, 'NorthAfrica' : 3}

winsI = modelingData['winsI'].to_numpy()
totals = modelingData['total'].to_numpy()

indexI = modelingData['regionI'].map(regionMap).to_numpy()
indexJ = modelingData['regionJ'].map(regionMap).to_numpy()

nRegions = len(regions)

print(indexI[:10])
print(indexJ[:10])


[0 0 0 1 1 2 0 0 0 1]
[1 2 3 2 3 3 1 2 3 2]


In [5]:
import pymc as pm

with pm.Model() as model:
    # Establish Bayesian prior, constrained to sum to 0
    beta = pm.ZeroSumNormal('beta', sigma=1.0, shape=nRegions)

    strengthI = beta[indexI]
    strengthJ = beta[indexJ]

    logOdds = strengthI - strengthJ
    probability = pm.math.sigmoid(logOdds)
     
    likelihood = pm.Binomial('likelihood', n=totals, p=probability, observed=winsI)

with model: 
    trace = pm.sample(1000, tune=1000, return_inferencedata=True)

Initializing NUTS using jitter+adapt_diag...
c:\Users\rcuhl\miniconda3\envs\georisk\Lib\site-packages\pytensor\link\c\cmodule.py:2986: UserWarning: PyTensor could not link to a BLAS installation. Operations that might benefit from BLAS will be severely degraded.
This usually happens when PyTensor is installed via pip. We recommend it be installed via conda/mamba/pixi instead.
Alternatively, you can use an experimental backend such as Numba or JAX that perform their own BLAS optimizations, by setting `pytensor.config.mode == 'NUMBA'` or passing `mode='NUMBA'` when compiling a PyTensor function.
For more options and details see https://pytensor.readthedocs.io/en/latest/troubleshooting.html#how-do-i-configure-test-my-blas-library
  warnings.warn(
Multiprocess sampling (4 chains in 4 jobs)
NUTS: [beta]


c:\Users\rcuhl\miniconda3\envs\georisk\Lib\site-packages\rich\live.py:260: UserWarning: install "ipywidgets" for 
Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

Sampling 4 chains for 1_000 tune and 1_000 draw iterations (4_000 + 4_000 draws total) took 25 seconds.


In [6]:
az.summary(trace, var_names=['beta'])

,mean,sd,hdi_3%,hdi_97%,mcse_mean,mcse_sd,ess_bulk,ess_tail,r_hat
beta[0],-0.686,0.001,-0.688,-0.685,0.0,0.0,5706.0,3078.0,1.0
beta[1],0.946,0.001,0.945,0.947,0.0,0.0,4164.0,3078.0,1.0
beta[2],0.849,0.001,0.848,0.850,0.0,0.0,4404.0,3099.0,1.0
beta[3],-1.109,0.001,-1.110,-1.107,0.0,0.0,2949.0,2958.0,1.0


In [7]:
samples = trace.posterior['beta'].values.reshape(-1, nRegions)

print(samples.shape)
print(samples[:5])

(4000, 4)
[[-0.68614081  0.94623687  0.84866333 -1.1087594 ]
 [-0.68669973  0.94633316  0.84863442 -1.10826785]
 [-0.68674361  0.94691339  0.84976066 -1.10993044]
 [-0.68680283  0.94625634  0.84924231 -1.10869582]
 [-0.68661715  0.9462067   0.84863736 -1.1082269 ]]


In [8]:
rankings = np.argsort(-samples, axis=1)
print(rankings[:10])

[[1 2 0 3]
 [1 2 0 3]
 [1 2 0 3]
 [1 2 0 3]
 [1 2 0 3]
 [1 2 0 3]
 [1 2 0 3]
 [1 2 0 3]
 [1 2 0 3]
 [1 2 0 3]]


In [9]:
uniqueRankings, counts = np.unique(rankings, axis=0, return_counts=True)
print(uniqueRankings)
print(counts)

[[1 2 0 3]]
[4000]


In [10]:
from scipy.stats import entropy

# Convert raw counts to proportions (that sum to 1)
probabilities = counts / counts.sum()
entropyValue = entropy(probabilities)

print(probabilities)
print(entropyValue)



[1.]
0.0


In [11]:
pairwiseFlip = (samples[:, 1] > samples[:, 2]).mean()
europeOverMiddleEast = pairwiseFlip

print(europeOverMiddleEast)

1.0


In [12]:
import itertools

flipProbabilities = dict()

for regionI, regionJ in itertools.combinations(range(nRegions), 2):
    probIOverJ = (samples[:, regionI] > samples[:, regionJ]).mean()
    pairLabel = f'{regions[regionI]} over {regions[regionJ]}'
    flipProbabilities[pairLabel] = probIOverJ

for pair, probability in flipProbabilities.items():
    print(pair,probability)



CaucasusCA over Europe 0.0
CaucasusCA over MiddleEast 0.0
CaucasusCA over NorthAfrica 1.0
Europe over MiddleEast 1.0
Europe over NorthAfrica 1.0
MiddleEast over NorthAfrica 1.0


In [13]:
hdiBounds = az.hdi(trace, var_names=['beta'])
print(hdiBounds)

<xarray.Dataset> Size: 144B
Dimensions:     (beta_dim_0: 4, hdi: 2)
Coordinates:
  * beta_dim_0  (beta_dim_0) int64 32B 0 1 2 3
  * hdi         (hdi) <U6 48B 'lower' 'higher'
Data variables:
    beta        (beta_dim_0, hdi) float64 64B -0.6877 -0.6849 ... -1.11 -1.107
Attributes:
    created_at:                 2026-07-05T15:12:06.349782+00:00
    arviz_version:              0.23.4
    inference_library:          pymc
    inference_library_version:  5.28.5
    sampling_time:              24.685869932174683
    tuning_steps:               1000


In [14]:
hdiArray = hdiBounds['beta'].values
print(hdiArray)

[[-0.68768    -0.6848808 ]
 [ 0.94497243  0.94722115]
 [ 0.84792385  0.85012049]
 [-1.11039596 -1.10722177]]


In [15]:
def intervalOverlap(lowerI, upperI, lowerJ, upperJ):
    return (lowerI <= upperJ) and (lowerJ <= upperI)

In [16]:
hdiOverlaps = dict()

for regionI, regionJ in itertools.combinations(range(nRegions), 2):
    lowerI, upperI = hdiArray[regionI]
    lowerJ, upperJ = hdiArray[regionJ]
    overlaps = intervalOverlap(lowerI, upperI, lowerJ, upperJ)
    pairLabel = f'{regions[regionI]} vs {regions[regionJ]}'
    hdiOverlaps[pairLabel] = overlaps

for pair, overlaps in hdiOverlaps.items():
    print(pair, overlaps)


CaucasusCA vs Europe False
CaucasusCA vs MiddleEast False
CaucasusCA vs NorthAfrica False
Europe vs MiddleEast False
Europe vs NorthAfrica False
MiddleEast vs NorthAfrica False


In [17]:
from Pipeline.uncertainty_metrics import computeUncertainty

results = computeUncertainty(trace, regions)
print(results['entropy'])
print(results['flipProbabilities'])
print(results['hdiOverlaps'])

0.0
{'CaucasusCA over Europe': np.float64(0.0), 'CaucasusCA over MiddleEast': np.float64(0.0), 'CaucasusCA over NorthAfrica': np.float64(1.0), 'Europe over MiddleEast': np.float64(1.0), 'Europe over NorthAfrica': np.float64(1.0), 'MiddleEast over NorthAfrica': np.float64(1.0)}
{'CaucasusCA vs Europe': np.False_, 'CaucasusCA vs MiddleEast': np.False_, 'CaucasusCA vs NorthAfrica': np.False_, 'Europe vs MiddleEast': np.False_, 'Europe vs NorthAfrica': np.False_, 'MiddleEast vs NorthAfrica': np.False_}


In [21]:
import pandas as pd

acledRaw = pd.read_csv('../Data/Raw/ACLED Data_2026-06-24.csv')

uniqueCountries = acledRaw['country'].nunique()
print(uniqueCountries)

countryCounts = acledRaw['country'].value_counts()
print(countryCounts)

83
country
Ukraine                  244529
Syria                    111325
Yemen                     78809
Palestine                 76999
Afghanistan               55147
                          ...  
Bailiwick of Guernsey        13
Isle of Man                  12
Faroe Islands                 8
Vatican City                  8
Gibraltar                     4
Name: count, Length: 83, dtype: int64


In [22]:
regionValues = acledRaw['region'].unique()
print(regionValues)

<ArrowStringArray>
['Northern Africa', 'Middle East', 'Europe', 'Caucasus and Central Asia']
Length: 4, dtype: str


In [24]:
regionalACLED = acledRaw[acledRaw['region'].isin(['Northern Africa', 'Middle East', 'Europe', 'Caucasus and Central Asia'])]

uniqueCountriesRegional = regionalACLED['country'].nunique()
print(uniqueCountriesRegional)

countryCountsRegional = regionalACLED.groupby('region')['country'].nunique()
print(countryCountsRegional)

83
region
Caucasus and Central Asia     9
Europe                       53
Middle East                  15
Northern Africa               6
Name: country, dtype: int64


In [27]:
countryEventCounts = regionalACLED.groupby('country').size().sort_values(ascending=False)
print(countryEventCounts.describe())
print(countryEventCounts.tail(20))

count        83.000000
mean      13098.445783
std       32366.510652
min           4.000000
25%         333.000000
50%        2230.000000
75%        7857.500000
max      244529.000000
dtype: float64
country
Iceland                  264
Luxembourg               208
Tajikistan               175
Kuwait                   111
Qatar                     62
Turkmenistan              51
Oman                      48
Greenland                 48
United Arab Emirates      45
San Marino                33
Andorra                   29
Monaco                    19
Akrotiri and Dhekelia     15
Bailiwick of Jersey       15
Liechtenstein             15
Bailiwick of Guernsey     13
Isle of Man               12
Faroe Islands              8
Vatican City               8
Gibraltar                  4
dtype: int64


In [28]:
survivingCountries = countryEventCounts[countryEventCounts >= 50].index
regionalACLEDFiltered = regionalACLED[regionalACLED['country'].isin(survivingCountries)]

print(regionalACLEDFiltered.groupby('region')['country'].nunique())

region
Caucasus and Central Asia     9
Europe                       41
Middle East                  13
Northern Africa               6
Name: country, dtype: int64


In [30]:
marketReturns = pd.read_csv('../Data/Processed/marketDF.csv', index_col=0, parse_dates=True)

print(marketReturns.head())
print(marketReturns.index.dtype)



            Close_EEM   Close_GLD  Close_ITA  Close_USO  Close_VIX
Date                                                              
2018-01-02  39.776558  125.150002  86.677139  96.559998       9.77
2018-01-03  40.157669  124.820000  86.805916  98.720001       9.15
2018-01-04  40.356510  125.459999  87.413139  98.959999       9.22
2018-01-05  40.704487  125.330002  88.199722  98.480003       9.22
2018-01-08  40.704487  125.309998  88.733322  99.040001       9.52
datetime64[us]


In [31]:
tradingDays = marketReturns.index
print(tradingDays[:5])

DatetimeIndex(['2018-01-02', '2018-01-03', '2018-01-04', '2018-01-05',
               '2018-01-08'],
              dtype='datetime64[us]', name='Date', freq=None)


In [34]:
from Pipeline.align_events import matchToTradingDay, getEventWindow

regionalACLEDFiltered['tradingDay'] = regionalACLEDFiltered['event_date'].apply(
    lambda eventDate: matchToTradingDay(eventDate, tradingDays)
)

print(regionalACLEDFiltered[['event_date', 'tradingDay']].head())

   event_date tradingDay
0  2018-01-01 2018-01-02
1  2018-01-01 2018-01-02
2  2018-01-01 2018-01-02
3  2018-01-01 2018-01-02
4  2018-01-01 2018-01-02
